# Nemotron: check the CPU export path before loading weights

**You are here → 06 — Nemotron with ExecuTorch.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| 0. Understand the request | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving and generated SQL, then evaluate the results to establish a quality baseline. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_optimization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| 5. CPU portability (optional) | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| **→ 6. Compatibility study (optional) (current)** | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |

Can the same **Nemotron 3 Nano 4B** hybrid checkpoint used in the serving chapters use the Qwen CPU deployment path? First test whether the installed ExecuTorch export recipe accepts its architecture.

Nemotron-H includes Mamba recurrent state alongside attention. Qwen's adapter does not supply that state mapping. The following cells execute a **fresh configuration-only probe** of the installed exporter. Unsupported configuration is a valid finding; this chapter does not run native inference or report generation timings.

Install the [notebook environment](../SETUP.md). Follow [the edge setup guide](../tutorial/EXECUTORCH_MAC.md), configure your edge Python, then run the cells in order. Downloads are opt-in; this check needs only the pinned `config.json`, not model weights.


## 1. Imports and editable configuration

`MODE="live"` performs the stages below. Choose **plan** to inspect without edge execution or **replay** to read the explicitly selected prior record. The notebook mode overrides an old local JSON mode. No stage installs packages or downloads weights implicitly.


In [1]:
from pathlib import Path
import os
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.edge_workflow import EdgeStudy

MODE = "live"  # "plan" previews stages; "replay" displays archived results
MODE = os.environ.get('LOCAL_AI_LAB_MODE', MODE)  # Explicit CPU validation override.
# Machine settings: tutorial/config/edge-nemotron.local.json
study = EdgeStudy('nemotron', root=ROOT, mode=MODE)
print('Effective mode:', study.mode)
print('Model:', 'nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16')
print('Hardware target: local CPU / separate ExecuTorch environment')
print('Session:', study.session_id)
print('Current-session measurements:', len(study.results))
display(JSON(study.config))


Effective mode: live
Model: nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16
Hardware target: local CPU / separate ExecuTorch environment
Session: 20261110T122034-8766f89a12
Current-session measurements: 0


<IPython.core.display.JSON object>

## 2. Preflight

Check the configured edge Python and installed runtime before any model work. Qwen requires a pinned source checkpoint and either a validated reusable export or explicit conversion/export. Nemotron needs only its pinned configuration. Missing prerequisites stop with instructions.


In [2]:
display(JSON(study.preflight()))


<IPython.core.display.JSON object>

## 3. Execute the compatibility probe

The next cell invokes [probe_nemotron_edge.py](../scripts/tutorial/probe_nemotron_edge.py) using your installed edge environment. It tests `nemotron_h` and `nemotron_3_nano_4b` against the installed model configuration enum and saves a unique result.

It reports exporter version, accepted model types and each attempt. If a recipe is accepted, conversion and native inference still require validation; acceptance alone supplies no performance result.


In [3]:
probe = study.probe()
from clinical_trial_mvp.tutorial.result_display import result_table
display(Markdown(result_table(probe)))


| Result | Status | SQL executed | Reference match | TTFT (s) | Request / inference (s) | Model load (s) | Peak memory (GiB) | Artifacts (MiB) |
|---|---|---|---|---|---|---|---|---|
| nemotron_h | blocked_compatibility | — | — | — | — | — | — | — |

| Model configuration | Accepted | Reason |
|---|---|---|
| nemotron_h | No | Invalid value 'nemotron_h', expected one of [stories110m, llama2, llama3, llama3_1, llama3_2, llama3_2_vision, static_llama, qwen2_5_0_5b, qwen2_5_1_5b, qwen2_5_coder_32b, qwen3_0_6b, qwen3_1_7b, qwen3_4b, qwen3_5_0_8b, qwen3_5_2b, qwen3_5_4b, phi_4_mini, smollm2, lfm2_350m, lfm2_700m, lfm2_1_2b, lfm2_5_350m, lfm2_5_1_2b]<br>    full_key: base.model_class<br>    reference_type=BaseConfig<br>    object_type=BaseConfig |
| nemotron_3_nano_4b | No | Invalid value 'nemotron_3_nano_4b', expected one of [stories110m, llama2, llama3, llama3_1, llama3_2, llama3_2_vision, static_llama, qwen2_5_0_5b, qwen2_5_1_5b, qwen2_5_coder_32b, qwen3_0_6b, qwen3_1_7b, qwen3_4b, qwen3_5_0_8b, qwen3_5_2b, qwen3_5_4b, phi_4_mini, smollm2, lfm2_350m, lfm2_700m, lfm2_1_2b, lfm2_5_350m, lfm2_5_1_2b]<br>    full_key: base.model_class<br>    reference_type=BaseConfig<br>    object_type=BaseConfig |

Native inference run: **No**.

| Session id | Session evidence path |
|---|---|
| 20261110T122034-8766f89a12 | experiment-tracker/tutorial/edge/sessions/20261110T122034-8766f89a12/probe-e8d2934e.json |

<details><summary>Full result details</summary>

<pre>{
  &quot;architecture&quot;: [
    &quot;NemotronHForCausalLM&quot;
  ],
  &quot;attempts&quot;: [
    {
      &quot;accepted&quot;: false,
      &quot;error_type&quot;: &quot;ValidationError&quot;,
      &quot;model_class&quot;: &quot;nemotron_h&quot;,
      &quot;reason&quot;: &quot;Invalid value &#x27;nemotron_h&#x27;, expected one of [stories110m, llama2, llama3, llama3_1, llama3_2, llama3_2_vision, static_llama, qwen2_5_0_5b, qwen2_5_1_5b, qwen2_5_coder_32b, qwen3_0_6b, qwen3_1_7b, qwen3_4b, qwen3_5_0_8b, qwen3_5_2b, qwen3_5_4b, phi_4_mini, smollm2, lfm2_350m, lfm2_700m, lfm2_1_2b, lfm2_5_350m, lfm2_5_1_2b]\n    full_key: base.model_class\n    reference_type=BaseConfig\n    object_type=BaseConfig&quot;
    },
    {
      &quot;accepted&quot;: false,
      &quot;error_type&quot;: &quot;ValidationError&quot;,
      &quot;model_class&quot;: &quot;nemotron_3_nano_4b&quot;,
      &quot;reason&quot;: &quot;Invalid value &#x27;nemotron_3_nano_4b&#x27;, expected one of [stories110m, llama2, llama3, llama3_1, llama3_2, llama3_2_vision, static_llama, qwen2_5_0_5b, qwen2_5_1_5b, qwen2_5_coder_32b, qwen3_0_6b, qwen3_1_7b, qwen3_4b, qwen3_5_0_8b, qwen3_5_2b, qwen3_5_4b, phi_4_mini, smollm2, lfm2_350m, lfm2_700m, lfm2_1_2b, lfm2_5_350m, lfm2_5_1_2b]\n    full_key: base.model_class\n    reference_type=BaseConfig\n    object_type=BaseConfig&quot;
    }
  ],
  &quot;command&quot;: [
    &quot;/Users/manimalak/Documents/Codex/local_clinical_trial/iterations/mvp3/publish/clinical-trial-mvp3-executed/.venv-edge/bin/python&quot;,
    &quot;/Users/manimalak/Documents/Codex/local_clinical_trial-mvp4 optimization/scripts/tutorial/probe_nemotron_edge.py&quot;,
    &quot;--output&quot;,
    &quot;/Users/manimalak/Documents/Codex/local_clinical_trial-mvp4 optimization/experiment-tracker/tutorial/edge/sessions/20261110T122034-8766f89a12/probe-raw-ebe74e45.json&quot;,
    &quot;--config&quot;,
    &quot;/Users/manimalak/Documents/Codex/local_clinical_trial-mvp4 optimization/models/edge-nemotron-4b/source/config.json&quot;
  ],
  &quot;config&quot;: {
    &quot;architectures&quot;: [
      &quot;NemotronHForCausalLM&quot;
    ],
    &quot;attention_bias&quot;: false,
    &quot;attention_dropout&quot;: 0.0,
    &quot;auto_map&quot;: {
      &quot;AutoConfig&quot;: &quot;configuration_nemotron_h.NemotronHConfig&quot;,
      &quot;AutoModelForCausalLM&quot;: &quot;modeling_nemotron_h.NemotronHForCausalLM&quot;
    },
    &quot;bos_token_id&quot;: 1,
    &quot;chunk_size&quot;: 256,
    &quot;conv_kernel&quot;: 4,
    &quot;eos_token_id&quot;: 2,
    &quot;expand&quot;: 2,
    &quot;head_dim&quot;: 128,
    &quot;hidden_dropout&quot;: 0.0,
    &quot;hidden_size&quot;: 3136,
    &quot;hybrid_override_pattern&quot;: &quot;M-M-M-MM-M-M*-M-M*-M-M-M*-M-M-MM*-MMM-M-M-&quot;,
    &quot;initializer_range&quot;: 0.02,
    &quot;intermediate_size&quot;: 12544,
    &quot;layer_norm_epsilon&quot;: 1e-05,
    &quot;mamba_head_dim&quot;: 80,
    &quot;mamba_hidden_act&quot;: &quot;silu&quot;,
    &quot;mamba_num_heads&quot;: 96,
    &quot;mamba_proj_bias&quot;: false,
    &quot;max_position_embeddings&quot;: 262144,
    &quot;mlp_bias&quot;: false,
    &quot;mlp_hidden_act&quot;: &quot;relu2&quot;,
    &quot;model_type&quot;: &quot;nemotron_h&quot;,
    &quot;n_groups&quot;: 8,
    &quot;num_attention_heads&quot;: 40,
    &quot;num_hidden_layers&quot;: 42,
    &quot;num_key_value_heads&quot;: 8,
    &quot;num_logits_to_keep&quot;: 1,
    &quot;pad_token_id&quot;: 0,
    &quot;rescale_prenorm_residual&quot;: true,
    &quot;residual_in_fp32&quot;: false,
    &quot;rms_norm_eps&quot;: 1e-05,
    &quot;sliding_window&quot;: null,
    &quot;ssm_state_size&quot;: 128,
    &quot;tie_word_embeddings&quot;: false,
    &quot;time_step_floor&quot;: 0.0001,
    &quot;time_step_max&quot;: 0.1,
    &quot;time_step_min&quot;: 0.001,
    &quot;time_step_rank&quot;: 256,
    &quot;torch_dtype&quot;: &quot;bfloat16&quot;,
    &quot;transformers_version&quot;: &quot;4.53.0&quot;,
    &quot;use_bias&quot;: false,
    &quot;use_cache&quot;: true,
    &quot;use_conv_bias&quot;: true,
    &quot;use_mamba_kernels&quot;: true,
    &quot;vocab_size&quot;: 131072
  },
  &quot;config_sha256&quot;: &quot;fde9241f66cd414458df444a80eb535f53aef2b4b240a91f092e651fa6f27219&quot;,
  &quot;end_to_end_seconds&quot;: null,
  &quot;evidence&quot;: &quot;local_export_configuration_probe&quot;,
  &quot;exporter_config_source_sha256&quot;: &quot;19ddd09846fb166e2dd06ac2205655ccadc2e40129dabe4f1d283cdfa9010072&quot;,
  &quot;graph_capture_attempted&quot;: false,
  &quot;hybrid_override_pattern&quot;: &quot;M-M-M-MM-M-M*-M-M*-M-M-M*-M-M-MM*-MMM-M-M-&quot;,
  &quot;installed_model_types&quot;: [
    &quot;stories110m&quot;,
    &quot;llama2&quot;,
    &quot;llama3&quot;,
    &quot;llama3_1&quot;,
    &quot;llama3_2&quot;,
    &quot;llama3_2_vision&quot;,
    &quot;static_llama&quot;,
    &quot;qwen2_5_0_5b&quot;,
    &quot;qwen2_5_1_5b&quot;,
    &quot;qwen2_5_coder_32b&quot;,
    &quot;qwen3_0_6b&quot;,
    &quot;qwen3_1_7b&quot;,
    &quot;qwen3_4b&quot;,
    &quot;qwen3_5_0_8b&quot;,
    &quot;qwen3_5_2b&quot;,
    &quot;qwen3_5_4b&quot;,
    &quot;phi_4_mini&quot;,
    &quot;smollm2&quot;,
    &quot;lfm2_350m&quot;,
    &quot;lfm2_700m&quot;,
    &quot;lfm2_1_2b&quot;,
    &quot;lfm2_5_350m&quot;,
    &quot;lfm2_5_1_2b&quot;
  ],
  &quot;measured_at&quot;: &quot;2026-11-10T12:20:37.509410+00:00&quot;,
  &quot;mode&quot;: &quot;live&quot;,
  &quot;model_id&quot;: &quot;nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16&quot;,
  &quot;model_revision&quot;: &quot;dfaf35de3e30f1867dd8dbc38a7fc9fb52d3914f&quot;,
  &quot;model_type&quot;: &quot;nemotron_h&quot;,
  &quot;native_inference_run&quot;: false,
  &quot;next_steps&quot;: [
    &quot;Implement or obtain a validated Nemotron-H export adapter preserving Mamba state and attention caches.&quot;,
    &quot;Compare converted FP32 logits with the pinned Hugging Face checkpoint.&quot;,
    &quot;Validate torch.export graph capture, backend/operator coverage, PTE load and native generation.&quot;,
    &quot;Then measure SQL correctness, TTFT, request latency and process memory separately from startup.&quot;
  ],
  &quot;peak_memory_bytes&quot;: null,
  &quot;pte_produced&quot;: false,
  &quot;raw_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T122034-8766f89a12/probe-raw-ebe74e45.json&quot;,
  &quot;scope&quot;: &quot;The installed standard LLM export recipe only; no claim that a custom Nemotron adapter is impossible.&quot;,
  &quot;session_evidence_path&quot;: &quot;experiment-tracker/tutorial/edge/sessions/20261110T122034-8766f89a12/probe-e8d2934e.json&quot;,
  &quot;session_id&quot;: &quot;20261110T122034-8766f89a12&quot;,
  &quot;source_identity&quot;: &quot;Caller-provided configuration; revision must be checked independently&quot;,
  &quot;sql_correctness&quot;: null,
  &quot;status&quot;: &quot;blocked_compatibility&quot;,
  &quot;ttft_seconds&quot;: null,
  &quot;versions&quot;: {
    &quot;executorch&quot;: &quot;1.5.1&quot;,
    &quot;omegaconf&quot;: &quot;2.3.1&quot;,
    &quot;torch&quot;: &quot;2.15.0&quot;,
    &quot;transformers&quot;: &quot;5.22.0&quot;
  },
  &quot;weights_downloaded&quot;: false
}</pre>
</details>

## 4. Interpret the finding

If the result is **blocked_compatibility**, the installed standard recipe lacks this model configuration. It does not establish that a custom port is impossible. A working port would need validated Mamba state handling, weight conversion, graph capture, supported operators, PTE loading and native generation before SQL or timing could be evaluated.


In [4]:
display(Markdown(study.report()))
from clinical_trial_mvp.tutorial.result_display import result_table
display(Markdown(result_table({'native_inference_run': probe.get('native_inference_run', False),
              'SQL_correctness': probe.get('sql_correctness'),
              'TTFT': probe.get('ttft_seconds'),
              'request_latency': probe.get('end_to_end_seconds'),
              'peak_memory': probe.get('peak_memory_bytes')})))


**Mode: live · Session: 20261110T122034-8766f89a12**

| Stage | Status | Evidence |
|---|---|---|
| probe | blocked_compatibility | experiment-tracker/tutorial/edge/sessions/20261110T122034-8766f89a12/probe-e8d2934e.json |

| Result | Status | SQL executed | Reference match | TTFT (s) | Request / inference (s) | Model load (s) | Peak memory (GiB) | Artifacts (MiB) |
|---|---|---|---|---|---|---|---|---|
| Result | — | — | — | — | — | — | — | — |

Native inference run: **No**.

<details><summary>Full result details</summary>

<pre>{
  &quot;SQL_correctness&quot;: null,
  &quot;TTFT&quot;: null,
  &quot;native_inference_run&quot;: false,
  &quot;peak_memory&quot;: null,
  &quot;request_latency&quot;: null
}</pre>
</details>

## Cleanup

The edge stages execute in foreground child processes and retain no serving process. Save the notebook to retain genuine current outputs. Failures stay in the session evidence.


In [5]:
print(study.close())


Edge stages run in foreground child processes; no serving process is retained.


## Reference: previously measured results

The [edge guide](../tutorial/EXECUTORCH_MAC.md) preserves the previous Qwen native run and Nemotron compatibility check. Choose `MODE="replay"` explicitly to inspect those records. The saved artifacts and historical records remain unchanged; they are not results from a new Live execution.
